###### Patricia Clarji 202500382 | Marie-Lie kadado 202402112

# Diabetes Readmission Prediction

## Objectives
- **Multiclass Classification:** Predict the patient's readmission category (`readmitted`)
- **Regression:** Predict time spent in hospital (`time_in_hospital`)

Dataset:
Diabetes 130-US Hospitals Dataset

### Project Description

Hospital readmissions are costly and may indicate complications in patient care.
The goal of this project is to analyze hospital records of diabetic patients and build machine learning models for:

- **Multiclass classification:** predict the patient's readmission category (`NO`, `>30`, `<30`)
- **Regression:** predict the patient's length of stay in the hospital (`time_in_hospital`)

The dataset contains hospital records of diabetic patients from 1999–2008 across 130 US hospitals.

### Step 1- Import Libraries and Models

In [ ]:
import pandas as pd
import numpy as np
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.patches import Rectangle

### Step 2- Load DataSet

In [ ]:
#Config:
DATA_PATH = "../data/diabetes.csv"
CLASSIFICATION_TARGET = "readmitted"
REGRESSION_TARGET = "time_in_hospital"
RANDOM_STATE = 42
TARGET= 'readmitted'

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH, na_values='?')

columns = df.columns

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Split Categorical and Numerical Columns
categorical_cols = [
    'race', 'gender', 'age', 'weight',
    'payer_code', 'medical_specialty',
    'diag_1', 'diag_2', 'diag_3',
    'max_glu_serum', 'A1Cresult',
    'metformin', 'repaglinide', 'nateglinide',
    'chlorpropamide', 'glimepiride', 'acetohexamide',
    'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'change', 'diabetesMed',
    'readmitted'
]

numerical_cols = [
    'encounter_id', 'patient_nbr',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

### Step 3- Dataset Overview

#### 3.1 Head (5 rows)

In [ ]:
print("Dataset preview:")
display(df.head(10))


#### 3.2 Info

In [ ]:
print("\nDataset info:")
df.info()

#### 3.3 Missing values

In [ ]:
print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False))

#### 3.4 Numerical cols

In [ ]:
print("\nNumerical summary:")
display(df[numerical_cols].describe().round(2))

#### 3.5 Categorical cols

In [ ]:
print("\nCategorical summary:")
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)

#### 3.6 Other

In [ ]:
print("\nShape of dataset:", df.shape)
print("\nNumber of numeric columns:", len(numerical_cols))
print("\nNumber of categorical columns:", len(categorical_cols))
print("\nClassification target:", CLASSIFICATION_TARGET)
print("\nRegression target:", REGRESSION_TARGET)

### Step 4- EDA Exploratory Data Analysis
In this step, we explore the dataset using visualizations to identify patterns, trends, and relationships between variables. This helps in understanding the data before applying preprocessing and machine learning models.

#### 4.0 Visualization SetUp

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

readmit_order = ['NO', '>30', '<30']
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
            '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']

num_cols = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_medications',
    'number_outpatient',
    'number_inpatient'
]

utilization_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

corr_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

#### 4.1 Target Variable Analysis
In this section, we analyze the target variables of the project: readmitted for the classification task and time_in_hospital for the regression task. This helps us understand class balance, distribution shape, and potential modeling challenges.

##### A- Readmission Distribution

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.countplot(data=df, x='readmitted', order=readmit_order)

total = len(df)
for p in ax.patches:
    if isinstance(p, Rectangle):
        count = int(p.get_height())
        pct = 100 * count / total
        ax.annotate(
            f"{count}\n({pct:.1f}%)",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha='center',
            va='bottom',
            fontsize=10
        )

plt.title("Distribution of Readmission Status")
plt.xlabel("Readmission Category")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()

#### Interpretation
Most patients were not readmitted, while the <30 class is the smallest. This shows class imbalance, which matters for the classification task.

#### B- time_in_hospital distribution
Next, we analyze the regression target time_in_hospital to understand its distribution, central tendency, and potential skewness or outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x='time_in_hospital', bins=14, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Time in Hospital")
axes[0].set_xlabel("Days in Hospital")
axes[0].set_ylabel("Frequency")

sns.boxplot(data=df, x='time_in_hospital', ax=axes[1])
axes[1].set_title("Boxplot of Time in Hospital")
axes[1].set_xlabel("Days in Hospital")

plt.tight_layout()
plt.show()

##### Interpretation

Most hospital stays are short, concentrated around a few days(peek btw 3-4 days), while a smaller number of cases have much longer stays. The boxplot also shows some outliers.

#### 4.2 Univariate Analysis

##### A- Numerical Features Distributions

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

##### B- Numerical Feature Outlier

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x=col, ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_xlabel(col)

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

##### C- Categorical Features Distribution

In [ ]:
cat_cols = ['gender', 'race', 'age']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = age_order if col == 'age' else df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

#### Interpretation
Patients who are readmitted tend to have longer hospital stays, more medications, and more prior inpatient visits. These variables strongly indicate patient complexity and are likely to be important predictors.

#### 4.3 Bivariate Analysis: Features vs Readmission

##### A- Demographic Features

In [ ]:
age_readmit = pd.crosstab(df['age'], df['readmitted'], normalize='index').reindex(age_order)
gender_order = df['gender'].value_counts().index
gender_readmit = pd.crosstab(df['gender'], df['readmitted'], normalize='index').reindex(gender_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set_title("Readmission Proportion by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Readmitted')

gender_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title("Readmission Proportion by Gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Proportion")
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Readmitted')

plt.tight_layout()
plt.show()

#### Interpretation
Older age groups show higher readmission proportions, suggesting age may be an important predictor. Gender differences are minimal, indicating that gender alone may not strongly influence readmission.

##### B- Clinical Features

In [ ]:
a1c_order = df['A1Cresult'].value_counts().index
a1c_readmit = pd.crosstab(df['A1Cresult'], df['readmitted'], normalize='index').reindex(a1c_order)

glu_order = df['max_glu_serum'].value_counts().index
glu_readmit = pd.crosstab(df['max_glu_serum'], df['readmitted'], normalize='index').reindex(glu_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

a1c_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set_title("Readmission Proportion by A1C Result")
axes[0].set_xlabel("A1C Result")
axes[0].set_ylabel("Proportion")
axes[0].legend(title='Readmitted')

glu_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title("Readmission Proportion by Max Glucose Serum")
axes[1].set_xlabel("Max Glucose Serum")
axes[1].set_ylabel("Proportion")
axes[1].legend(title='Readmitted')

plt.tight_layout()
plt.show()

#### Interpretation
Patients with abnormal A1C and glucose results tend to have higher readmission proportions. This suggests that poor glycemic control is associated with increased readmission risk.

#### C- Hospital Utilization Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

sns.boxplot(data=df, x='readmitted', y='time_in_hospital', order=readmit_order, ax=axes[0])
axes[0].set_title("Time in Hospital by Readmission Class")
axes[0].set_xlabel("Readmission Category")
axes[0].set_ylabel("Days in Hospital")

sns.boxplot(data=df, x='readmitted', y='num_medications', order=readmit_order, ax=axes[1])
axes[1].set_title("Number of Medications by Readmission Class")
axes[1].set_xlabel("Readmission Category")
axes[1].set_ylabel("Number of Medications")

sns.boxplot(data=df, x='readmitted', y='number_inpatient', order=readmit_order, ax=axes[2])
axes[2].set_title("Previous Inpatient Visits by Readmission Class")
axes[2].set_xlabel("Readmission Category")
axes[2].set_ylabel("Number of Inpatient Visits")

sns.boxplot(data=df, x='readmitted', y='num_lab_procedures', order=readmit_order, ax=axes[3])
axes[3].set_title("Lab Procedures by Readmission Class")
axes[3].set_xlabel("Readmission Category")
axes[3].set_ylabel("Number of Lab Procedures")

plt.tight_layout()
plt.show()

#### Interpretation
Patients who are readmitted tend to have longer hospital stays, use more medications, and have more previous inpatient visits. These patterns suggest that hospital utilization and clinical complexity are strongly related to readmission risk.

#### 4.4. Missingness Overview

##### A- Missing percentage by feature

In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(10, 6))
sns.barplot(x=missing_pct.values, y=missing_pct.index)
plt.title("Missing Values Percentage by Feature")
plt.xlabel("Missing Percentage")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

##### B- Missingness heatmap for columns with missing values

In [ ]:
missing_cols = df.columns[df.isnull().any()].tolist()

plt.figure(figsize=(12, 6))
sns.heatmap(df[missing_cols].isnull(), cbar=False)
plt.title("Missingness Heatmap")
plt.xlabel("Features with Missing Values")
plt.ylabel("Observations")
plt.tight_layout()
plt.show()

#### 4.5 Correlation and Numerical Relationships

##### A- Correlation heatmap

In [ ]:
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

#### Interpretation

The scatterplots show moderate positive relationships between several numerical features. In particular, patients with more diagnoses and lab procedures tend to receive more medications. While these relationships are not extremely strong, they indicate that combinations of these variables may be useful for predicting readmission. No severe linear relationships are observed, suggesting that multicollinearity may not be a major issue.

##### B- Selected numerical relationship plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=df, x='num_lab_procedures', y='num_medications', alpha=0.3, ax=axes[0])
axes[0].set_title("Lab Procedures vs Medications")

sns.scatterplot(data=df, x='number_inpatient', y='time_in_hospital', alpha=0.3, ax=axes[1])
axes[1].set_title("Previous Inpatient Visits vs Hospital Stay")

sns.scatterplot(data=df, x='number_diagnoses', y='num_medications', alpha=0.3, ax=axes[2])
axes[2].set_title("Diagnoses vs Medications")

plt.tight_layout()
plt.show()

#### Interpretation

The scatterplots show moderate positive relationships between several numerical features. In particular, patients with more diagnoses and lab procedures tend to receive more medications. While these relationships are not extremely strong, they indicate that combinations of these variables may be useful for predicting readmission. No severe linear relationships are observed, suggesting that multicollinearity may not be a major issue.

#### 4.6 Additional Categorical Analysis

##### Readmission rate by top race categories

In [ ]:
top_race = df['race'].value_counts().index
race_readmit = pd.crosstab(df['race'], df['readmitted'], normalize='index').reindex(top_race)

plt.figure(figsize=(8, 5))
race_readmit[readmit_order].plot(kind='bar', stacked=True)
plt.title("Readmission Proportion by Race")
plt.xlabel("Race")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

##### Readmission rate by admission type

In [ ]:
admission_order = df['admission_type_id'].value_counts().index
admission_readmit = pd.crosstab(df['admission_type_id'], df['readmitted'], normalize='index').reindex(admission_order)

plt.figure(figsize=(8, 5))
admission_readmit[readmit_order].plot(kind='bar', stacked=True)
plt.title("Readmission Proportion by Admission Type")
plt.xlabel("Admission Type ID")
plt.ylabel("Proportion")
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

### Step 5- Data Preprocessing

Data preprocessing prepares the dataset for machine learning by handling
missing values, removing irrelevant features, encoding categorical variables,
and scaling numerical features.

#### 5.1 Missing Values Check

In [ ]:
print("Missing values per column:")
display(df.isnull().sum().sort_values(ascending=False))

### Outlier Handling

Although outliers were observed in several numerical features, they were not removed.

In healthcare datasets, extreme values often represent real clinical conditions rather than errors.

Removing them could result in loss of important information, especially for severe patient cases.

Therefore, outliers were retained to preserve data integrity.

##### 5.1.1 Handling missing values

Different imputation methods were used depending on the nature of each feature:

- **Categorical medical variables (diagnosis, lab tests):**
  - Missing values were treated as meaningful categories (e.g., "Missing", "NotMeasured") since absence of measurement carries information.
  - Rare race categories such as Asian and Hispanic were grouped into the "Other" category to reduce sparsity and improve model generalization. This helps prevent the model from overfitting to categories with very few observations.

- **Medication features:**
  Missing values were replaced with "No", assuming the medication was not prescribed.

- **Numerical features:**
  - Median was used for skewed distributions to reduce the impact of outliers.
  - Zero was used for count-based features where missing implies no occurrences.

- **Target variable:**
  Rows with missing target values were removed to ensure valid supervised learning.

This approach preserves domain meaning and avoids introducing bias.

In [ ]:
# 1. Remove rows with missing target
df = df.dropna(subset=['readmitted'])

# 2. Diagnosis columns -> keep missing as category
missing_category_cols = ['race', 'diag_1', 'diag_2', 'diag_3']

for col in missing_category_cols:
    df[col] = df[col].fillna('Missing').astype(str)

# Combine rare race categories into "Other"
df['race'] = df['race'].replace({
    'Asian': 'Other',
    'Hispanic': 'Other'
})


# 3. Lab result columns -> missing means not measured
lab_cols = ['max_glu_serum', 'A1Cresult']
for col in lab_cols:
    df[col] = df[col].fillna('NotMeasured')

# 4. Medication columns -> missing means No
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

for col in medication_cols:
    df[col] = df[col].fillna('No')

# 5. Binary columns
df['change'] = df['change'].fillna('No')
df['diabetesMed'] = df['diabetesMed'].fillna('No')

# 6. Hospital code columns should be categorical
coded_cols = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in coded_cols:
    df[col] = df[col].astype(str)

# 7. Numeric columns
median_cols = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_diagnoses'
]

zero_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

for col in zero_cols:
    df[col] = df[col].fillna(0)

print("Remaining missing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

#### 5.2 Drop Columns with Too Many Missing Values

In [ ]:
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
df.drop(columns=cols_to_drop, inplace=True)

print("Dataset shape after dropping columns:", df.shape)

#### 5.3 Remove Duplicate Rows

In [ ]:
print("Duplicate rows before:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Duplicate rows after:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

#### 5.4 Clean Invalid Gender Values

In [ ]:
print("Gender values before cleaning:")
print(df['gender'].value_counts(dropna=False))

df = df[df['gender'].isin(['Male', 'Female'])]

print("\nGender values after cleaning:")
print(df['gender'].value_counts(dropna=False))

#### 5.5 Check Multiclass Target Distribution

In [ ]:
print("Readmission classes:")
print(df['readmitted'].value_counts(dropna=False))

#### 5.6 Convert Age Groups to Numeric Values

In [ ]:
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}

df['age_numeric'] = df['age'].map(age_map)

display(df[['age', 'age_numeric']].head())

#### 5.7 Drop ID Columns

In [ ]:
id_cols = ['encounter_id', 'patient_nbr']
print("ID columns:", id_cols)

df.drop(columns=id_cols, inplace=True)

print("Dropped ID columns.")

#### 5.8 Final Missing Values Check

In [ ]:
print("Remaining missing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

#### 5.9 Preprocessing Summary

In [ ]:
print("Final dataset shape:", df.shape)
print("Remaining missing values total:", df.isnull().sum().sum())

### Step-6- Multiclass Classification
We now build models to predict the readmission category (readmitted).

#### 6.0 Import Classification Libraries

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#### 6.1 Prepare Features and Target

In this step:
    Separate features (X) and target (y)
    Ensure the target is correctly defined
    Prepare the dataset for modeling

In [ ]:
y_clf = df["readmitted"]
X_clf = df.drop(columns=["readmitted"])

print("Feature matrix shape:", X_clf.shape)
print("Target shape:", y_clf.shape)
print("\nTarget distribution:")
print(y_clf.value_counts())

#### 6.2 Train-Test Split
We split the dataset into training and testing sets.
Since this is classification, we use stratification to preserve class balance.

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

print("Training set size:", X_train_clf.shape)
print("Testing set size:", X_test_clf.shape)

#### 6.3 Encoding Categorical Features

Categorical variables were encoded using One-Hot Encoding since they do not have an inherent ordinal relationship.

This allows models to treat each category independently.

However, due to the large number of categorical features, this significantly increases dimensionality, which can impact computational efficiency.

In [ ]:
categorical_features = X_train_clf.select_dtypes(include=["object"]).columns.tolist()

X_train_clf_encoded = pd.get_dummies(
    X_train_clf,
    columns=categorical_features,
    drop_first=True
)

X_test_clf_encoded = pd.get_dummies(
    X_test_clf,
    columns=categorical_features,
    drop_first=True
)

X_train_clf_encoded, X_test_clf_encoded = X_train_clf_encoded.align(
    X_test_clf_encoded,
    join="left",
    axis=1,
    fill_value=0
)

print("Encoded training shape:", X_train_clf_encoded.shape)
print("Encoded testing shape:", X_test_clf_encoded.shape)

#### 6.4 Feature Scaling

In [ ]:
scaler_clf = StandardScaler(with_mean=False)

X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf_encoded)
X_test_clf_scaled = scaler_clf.transform(X_test_clf_encoded)

print("Feature scaling completed.")

#### 6.5 Train and evaluate Multiple Classification Models

In this step, we train several classification models and compare their performance.

We will use:

- K-Nearest Neighbors (KNN)
- Support Vector Machine (Linear SVM)
- Decision Tree
- Random Forest
- Naive Bayes

Logistic Regression was considered as a baseline model, but it was excluded because it required a long training time on the high-dimensional one-hot encoded dataset.

##### Evaluate model definition

In [ ]:
def run_evaluate_model(model, model_name, X_train, X_test, y_train, y_test):

    print(f"\n--- {model_name} ---")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred) * 100
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy: {acc:.2f}%")
    print(f"Weighted F1 Score: {f1:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return acc, f1, y_pred

#### Logistic Regression
Logistic Regression was considered as a baseline model, but it was excluded from the final comparison because it required a long training time on the high-dimensional one-hot encoded dataset.

####  KNN
KNN is sensitive to high-dimensional data, and performance decreases after one-hot encoding due to sparsity.

In [ ]:
knn_model = KNeighborsClassifier(
    n_neighbors=15,
    weights="distance",
    n_jobs=-1
)

acc_knn, f1_knn, y_pred_knn = run_evaluate_model(
    knn_model,
    "KNN",
    X_train_clf_scaled,
    X_test_clf_scaled,
    y_train_clf,
    y_test_clf
)

#### Linear SVM

In [ ]:
svm_model = LinearSVC(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=3000
)

acc_svm, f1_svm, y_pred_svm = run_evaluate_model(
    svm_model,
    "Linear SVM",
    X_train_clf_scaled,
    X_test_clf_scaled,
    y_train_clf,
    y_test_clf
)

#### Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=18,
    min_samples_split=10,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

acc_tree, f1_tree, y_pred_tree = run_evaluate_model(
    tree_model,
    "Decision Tree",
    X_train_clf_encoded,
    X_test_clf_encoded,
    y_train_clf,
    y_test_clf
)

#### Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=3,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

acc_rf, f1_rf, y_pred_rf = run_evaluate_model(
    rf_model,
    "Random Forest",
    X_train_clf_encoded,
    X_test_clf_encoded,
    y_train_clf,
    y_test_clf
)

#### Naive Bayes

In [ ]:
nb_model = GaussianNB()

acc_nb, f1_nb, y_pred_nb = run_evaluate_model(
    nb_model,
    "Naive Bayes",
    X_train_clf_encoded,
    X_test_clf_encoded,
    y_train_clf,
    y_test_clf
)

#### 6.6 Compare Models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "KNN",
        "Linear SVM",
        "Decision Tree",
        "Random Forest",
        "Naive Bayes"
    ],
    "Accuracy (%)": [
        acc_knn,
        acc_svm,
        acc_tree,
        acc_rf,
        acc_nb
    ],
    "F1 Score": [
        f1_knn,
        f1_svm,
        f1_tree,
        f1_rf,
        f1_nb
    ]
})

results = results.sort_values(by="F1 Score", ascending=False)

display(results)

plt.figure(figsize=(8, 5))
sns.barplot(data=results, x="F1 Score", y="Model")
plt.title("Model Comparison Based on Weighted F1 Score")
plt.xlabel("Weighted F1 Score")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

#### 6.7 Best Model Analysis
Let’s analyze the best-performing model.

In [ ]:
best_model_name = results.iloc[0]["Model"]

if best_model_name == "KNN":
    y_pred_best = y_pred_knn
elif best_model_name == "Linear SVM":
    y_pred_best = y_pred_svm
elif best_model_name == "Decision Tree":
    y_pred_best = y_pred_tree
elif best_model_name == "Random Forest":
    y_pred_best = y_pred_rf
else:
    y_pred_best = y_pred_nb

print("Best model:", best_model_name)

cm = confusion_matrix(y_test_clf, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=['NO', '>30', '<30'],
            yticklabels=['NO', '>30', '<30'])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print("Classification Report:\n")
print(classification_report(y_test_clf, y_pred_best, zero_division=0))

#### Decision Tree Implementation
A visualization of the decision tree was generated to understand how the model makes decisions.

Due to the high dimensionality of the dataset, only the first few levels of the tree were displayed. These levels represent the most important decision rules used by the model.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))

plot_tree(
    tree_model,
    max_depth=3,
    feature_names=X_train_clf_encoded.columns,
    class_names=['NO', '>30', '<30'],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree Visualization (First 3 Levels)")
plt.show()

#### 6.8 Classification Conclusion

#### 6.8 Classification Conclusion

The classification models were evaluated using both accuracy and weighted F1-score, with greater emphasis on the F1-score due to the class imbalance in the readmission variable.

Among the tested models, tree-based approaches such as Random Forest and Decision Tree achieved the best performance. This is expected, as these models are capable of capturing complex and non-linear relationships within the data.

K-Nearest Neighbors (KNN) showed lower performance, which can be explained by the high-dimensional feature space created after one-hot encoding. Distance-based models are less effective in such sparse representations.

Linear SVM provided stable performance but was limited in capturing complex patterns compared to tree-based models.

Overall, the classification performance is moderate. This is expected given the nature of the dataset, which includes class imbalance, high dimensionality, and relatively weak relationships between some features and the target variable.

Despite these challenges, the models were able to identify meaningful patterns, particularly from features related to hospital utilization, patient complexity, and clinical measurements.

### Step-7- Regression Modeling
We now build models to predict the time spent in the hospital category (time_in_hospital).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,  StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

#### 7.1 Prepare Features and Target

We separate features (X) and target (y), we make sure the target is correctly defined and we prepare the dataset for modeling

In [ ]:
# Define target variable
REGRESSION_TARGET = 'time_in_hospital'
y_reg = df[REGRESSION_TARGET]

# Define feature matrix
X_reg = df.drop(columns=[REGRESSION_TARGET])

print("Feature matrix shape:", X_reg.shape)
print("Target shape:", y_reg.shape)

print("\nTarget distribution:")
print(y_reg.value_counts())

#### 7.2 Encode Categorical Features
Machine learning models require numerical input, so we use hot one encoding to convert categorical data.

In [ ]:
# Identify categorical columns (excluding target)
categorical_features = X_reg.select_dtypes(include=['object']).columns

print("Categorical features to encode:", len(categorical_features))

# One-hot encoding
X_reg_encoded = pd.get_dummies(X_reg, columns=categorical_features.tolist(), drop_first=True)

print("Encoded feature matrix shape:", X_reg_encoded.shape)

#### 7.3 Train-Test Split

Split dataset into training and testing sets.

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg_encoded, y_reg, test_size=0.2, random_state=42
)

print("Training set size:", X_train_reg.shape)
print("Testing set size:", X_test_reg.shape)

#### 7.4 Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)

print("Feature scaling completed.")

#### 7.5 Train Regression Models

We will compare a set of regression algorithms.

We will use:
- Linear-family models
    - Linear Regression
    - Ridge

- Tree-based and ensemble models
    - Random Forest Regressor

### Initialize Models

In [ ]:
models_reg = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE)
}

### Train Models

In [ ]:
trained_models_reg = {}

for name, model in models_reg.items():

    if name in ["Linear Regression", "Ridge"]:
        model.fit(X_train_reg_scaled, y_train_reg)
    else:
        model.fit(X_train_reg, y_train_reg)

    trained_models_reg[name] = model

#### 7.6 Evaluate Regression Models

We evaluate models using:
mean_squared_error and r2_score

In [ ]:
results_reg = []

for name, model in trained_models_reg.items():

    # Handle scaling
    if name in ["Linear Regression", "Ridge"]:
        y_pred = model.predict(X_test_reg_scaled)
    else:
        y_pred = model.predict(X_test_reg)

    # Metrics
    mse = mean_squared_error(y_test_reg, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_reg, y_pred)

    results_reg.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_reg_df = pd.DataFrame(results_reg).sort_values(by="R2 Score", ascending=False)

display(results_reg_df)

#### 7.7 Compare Models

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=results_reg_df, x="R2 Score", y="Model")

plt.title("Regression Model Comparison (R²)")
plt.xlabel("R² Score")
plt.ylabel("Model")

plt.tight_layout()
plt.show()

In [ ]:
cv_results_reg = []

for name, model in models_reg.items():
    if name in ["Linear Regression", "Ridge"]:
        scores = cross_val_score(model, X_train_reg_scaled, y_train_reg, cv=5, scoring='r2')
    else:
        scores = cross_val_score(model, X_train_reg, y_train_reg, cv=5, scoring='r2')

    cv_results_reg.append({
        "Model": name,
        "CV R2 Mean": scores.mean(),
        "CV Std": scores.std()
    })

cv_reg_df = pd.DataFrame(cv_results_reg).sort_values(by="CV R2 Mean", ascending=False)

display(cv_reg_df)

#### 7.8 Best Model Analysis
Now we analyze the best-performing model.

##### Get Best Model

In [ ]:
best_model_name = results_reg_df.iloc[0]["Model"]
best_model = trained_models_reg[best_model_name]

print("Best regression model:", best_model_name)

##### Best Model Analysis

In [ ]:
# Predict using best model
if best_model_name in ["Linear Regression", "Ridge"]:
    y_pred_best = best_model.predict(X_test_reg_scaled)
else:
    y_pred_best = best_model.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_best)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_best)

print("Best Model:", best_model_name)
print("R2:", r2)
print("RMSE:", rmse)

plt.scatter(y_test_reg, y_pred_best)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.show()

#### 7.9 Regression Conclusion

Among the evaluated models, **[Best Model Name]** achieved the highest R² score.

- The model shows the best performance in predicting the target variable.
- Linear Regression and Ridge produced similar results, suggesting a mostly linear relationship in the data.
- Performance can be further improved with better preprocessing or tuning.